# 08 - Merge Dataset Oleh-Oleh

Notebook ini digunakan untuk menggabungkan dataset oleh-oleh, membersihkan duplikasi, menghapus data hotel yang tidak sesuai, dan menyimpan dataset akhir `oleh_oleh_final.csv`.

In [ ]:
import re
from pathlib import Path

import pandas as pd

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = [
    RAW_DIR / "oleh oleh2.csv",
    RAW_DIR / "oleholeh3.csv",
]

OUTPUT_FILE = PROCESSED_DIR / "oleh_oleh_final.csv"
REQUIRED_COLUMNS = ["name", "category", "rating", "address", "subtypes"]

In [ ]:
# Load kedua dataset oleh-oleh.
raw_frames = []
for path in SOURCE_FILES:
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip().str.lower()
    raw_frames.append(frame)
    print(f"{path.name}: {frame.shape[0]} baris, {frame.shape[1]} kolom")

df_raw = pd.concat(raw_frames, ignore_index=True)
print("Total sebelum cleaning:", df_raw.shape)

In [ ]:
# Validasi kolom penting sebelum proses cleaning.
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df_raw.columns))
if missing_columns:
    raise ValueError(f"Kolom wajib belum tersedia: {missing_columns}")

print("Kolom penting tersedia:", REQUIRED_COLUMNS)
df_raw[REQUIRED_COLUMNS].head()

In [ ]:
def normalize_text(value):
    """Membersihkan spasi berlebih dan nilai kosong."""
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def normalize_key(value):
    """Membuat versi nama yang stabil untuk pengecekan duplikat."""
    text = normalize_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def contains_hotel_keyword(row):
    """Mendeteksi data hotel/penginapan yang masuk ke dataset oleh-oleh."""
    keywords = [
        "hotel", "homestay", "guest house", "guesthouse", "villa",
        "resort", "hostel", "penginapan", "losmen", "inn"
    ]
    combined_text = " ".join(
        normalize_text(row.get(column, ""))
        for column in ["name", "category", "subtypes", "type"]
    ).lower()
    return any(keyword in combined_text for keyword in keywords)

In [ ]:
# Merge, normalisasi teks, hapus kontaminasi hotel, dan hapus duplikat tempat.
df_oleholeh = df_raw.copy()

for column in REQUIRED_COLUMNS:
    df_oleholeh[column] = df_oleholeh[column].apply(normalize_text)

df_oleholeh["name_key"] = df_oleholeh["name"].apply(normalize_key)
duplicate_before = int(df_oleholeh.duplicated(subset=["name_key"]).sum())

hotel_mask = df_oleholeh.apply(contains_hotel_keyword, axis=1)
hotel_contamination_count = int(hotel_mask.sum())
df_oleholeh = df_oleholeh.loc[~hotel_mask].copy()

# Baris tanpa nama tidak aman untuk rekomendasi, sehingga dihapus.
before_critical_drop = len(df_oleholeh)
df_oleholeh = df_oleholeh[df_oleholeh["name_key"] != ""].copy()
critical_rows_removed = before_critical_drop - len(df_oleholeh)

# Nilai kosong non-kritis dibuat eksplisit agar mudah dijelaskan.
df_oleholeh["category"] = df_oleholeh["category"].replace("", "unknown")
df_oleholeh["subtypes"] = df_oleholeh["subtypes"].replace("", "unknown")
df_oleholeh["address"] = df_oleholeh["address"].replace("", "unknown")
df_oleholeh["rating"] = pd.to_numeric(df_oleholeh["rating"], errors="coerce").fillna(0).clip(0, 5)

before_dedup = len(df_oleholeh)
df_oleholeh = df_oleholeh.drop_duplicates(subset=["name_key"], keep="first").copy()
duplicates_removed = before_dedup - len(df_oleholeh)

# Label tipe diseragamkan untuk master dataset.
df_oleholeh["type"] = "oleh_oleh"
df_oleholeh = df_oleholeh.drop(columns=["name_key"]).reset_index(drop=True)

summary = pd.DataFrame({
    "metric": [
        "total_raw",
        "duplicate_name_before_cleaning",
        "hotel_contamination_removed",
        "critical_rows_removed",
        "duplicates_removed_after_filter",
        "final_rows",
        "final_columns",
    ],
    "value": [
        len(df_raw),
        duplicate_before,
        hotel_contamination_count,
        critical_rows_removed,
        duplicates_removed,
        df_oleholeh.shape[0],
        df_oleholeh.shape[1],
    ]
})
summary

In [ ]:
# Ringkasan validasi untuk dokumentasi Bab 3.
missing_summary = df_oleholeh[REQUIRED_COLUMNS + ["type"]].isna().sum().reset_index()
missing_summary.columns = ["column", "missing_count"]

duplicate_count = int(df_oleholeh.duplicated(subset=["name"]).sum())

print("Dataset shape:", df_oleholeh.shape)
print("Duplicate count by name:", duplicate_count)
print("Type counts:")
print(df_oleholeh["type"].value_counts())

missing_summary

In [ ]:
# Contoh data akhir yang sudah bersih.
sample_columns = ["name", "category", "rating", "address", "subtypes", "type"]
df_oleholeh[sample_columns].head(10)

In [ ]:
# Simpan dataset final oleh-oleh.
df_oleholeh.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")